# Does smoothing Location+ around each pitch's own location matter?

`bestpitch.ipynb`/`bestpitch.py` already target-average both candidates and the
actual pitch (via `_actual_smoothed_pitching_plus`) around a *zone reference
point*, but only as a private quantity used solely for the bestPitch+
subtraction. The "official" `location_run_value`/`pitch_pitching_plus` that
everything else (leaderboards, `pitching.ipynb`) reports is still a pinpoint
prediction at each pitch's own exact `(plate_x, plate_z_rel)`.

Before touching `location.py`'s out-of-fold training loop to make that official
number target-averaged too (a real rewrite: `cross_val_predict` doesn't expose
a hook to query extra points through each fold's held-out model), this notebook
checks whether it would actually change anything that matters, on a sample,
using the already-cached final models:

1. **Per-pitch**: does target-averaging around a pitch's own location noticeably
   change its score, or move it any closer to (or further from) the real
   outcome (`delta_pitcher_run_exp`)?
2. **Aggregate**: does it change the season-level `location_plus`/leaderboard
   numbers that averaging over many pitches already stabilizes? Or does
   per-pitch smoothing only matter for `bestPitch+`'s *max*-over-candidates
   search, which has no such built-in noise cancellation?

Reuses `bestpitch.py`'s own `_predict_at_point` directly, centered on each
pitch's own location instead of a shared zone reference. Nothing reimplemented.

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pitching_plus").is_dir():
            return candidate
    raise RuntimeError("Could not locate repo root (expected a 'pitching_plus' directory in a parent).")


REPO_ROOT = find_repo_root(Path.cwd())
SCRIPTS_DIR = REPO_ROOT / "pitching_plus" / "scripts"
DATA_PATH = REPO_ROOT / "data" / "MLB_2021-2025.csv"

sys.path.insert(0, str(SCRIPTS_DIR))

import stuff as stuff_mod
import location as location_mod
from bestpitch import _predict_at_point, TARGET_RADIUS_FT
from location import LOCATION_FEATURES, TARGET_COL, PITCH_TYPE_COL, PITCHER_COL, SEASON_COL

RNG_SEED = 0
SAMPLE_SIZE = 300_000

## Load a sample and the cached models

Same `usecols`/in-scope filtering as `location.py` itself. Loads the full
column-trimmed CSV (cheap, ~45s, matching `fg_pitching.ipynb`'s own load time),
then samples `SAMPLE_SIZE` in-scope pitches for the comparison, rather than
scoring all 3.5M. This is a diagnostic, not a production run.

In [ ]:
t0 = time.time()
usecols = sorted(set(location_mod.REQUIRED_COLS) | set(location_mod.BASE_STATE_COLS))
raw_df = pd.read_csv(DATA_PATH, usecols=usecols)
print(f"Loaded {len(raw_df):,} raw pitches in {time.time() - t0:.0f}s")

in_scope = (
    raw_df.loc[~raw_df[PITCH_TYPE_COL].isin(stuff_mod.JUNK_PITCH_TYPES)]
    .dropna(subset=location_mod.REQUIRED_COLS)
)
engineered = location_mod._build_features(in_scope)
engineered[TARGET_COL] = in_scope[TARGET_COL].to_numpy()

sample = engineered.sample(n=SAMPLE_SIZE, random_state=RNG_SEED)
print(f"{len(engineered):,} in-scope pitches; sampled {len(sample):,} for this check")

models = location_mod.load_cached_models()
print(f"{len(models)} cached per-pitch-type models: {sorted(models.keys())}")

## Pinpoint vs. own-location target-averaged scoring

Pinpoint: the model's direct prediction at the pitch's own exact location
(what `location_run_value` is today). Smoothed: `_predict_at_point` queried at
5 points (center plus N/S/E/W within `TARGET_RADIUS_FT`) centered on that same
pitch's own location, not a zone reference, since a real pitch has a genuine
location worth keeping, unlike a candidate. `_predict_at_point` already accepts
per-row arrays for `plate_x`/`plate_z_rel`, so this needs no changes to it,
just per-row centers instead of one shared scalar.

In [ ]:
non_location_features = [c for c in LOCATION_FEATURES if c not in ("plate_x", "plate_z_rel", "plate_x_armside")]

def target_averaged_own_location(base, model):
    zone_height = (base["sz_top"] - base["sz_bot"]).to_numpy()
    z_offset = TARGET_RADIUS_FT / zone_height
    px = base["plate_x"].to_numpy()
    pz = base["plate_z_rel"].to_numpy()

    points = [
        (px, pz),
        (px + TARGET_RADIUS_FT, pz),
        (px - TARGET_RADIUS_FT, pz),
        (px, pz + z_offset),
        (px, pz - z_offset),
    ]
    predictions = [_predict_at_point(base, model, non_location_features, p_x, p_z) for p_x, p_z in points]
    return np.mean(predictions, axis=0)


pinpoint = pd.Series(np.nan, index=sample.index, dtype=float)
smoothed = pd.Series(np.nan, index=sample.index, dtype=float)

t0 = time.time()
for ptype, model in models.items():
    rows = sample[PITCH_TYPE_COL] == ptype
    if not rows.any():
        continue
    base = sample.loc[rows]
    pinpoint.loc[rows] = model.predict(base[LOCATION_FEATURES].to_numpy())
    smoothed.loc[rows] = target_averaged_own_location(base, model)

sample = sample.assign(pinpoint=pinpoint, smoothed=smoothed).dropna(subset=["pinpoint", "smoothed"])
print(f"Scored {len(sample):,} pitches in {time.time() - t0:.0f}s")

## Q1: per-pitch. Does smoothing change individual scores or outcome-correlation?

Both predictions come from the same in-sample cached model, not OOF. That's
fine for an *isolated* pinpoint-vs-smoothed comparison, since any in-sample
optimism affects both sides equally, but it's not a stand-in for what
production `location_run_value`'s actual (OOF) magnitude would be.

In [ ]:
diff = sample["smoothed"] - sample["pinpoint"]
print("Per-pitch difference (smoothed - pinpoint):")
print(diff.describe().round(4).to_string())
print()

corr_pinpoint = sample["pinpoint"].corr(sample[TARGET_COL])
corr_smoothed = sample["smoothed"].corr(sample[TARGET_COL])
print(f"corr(pinpoint, {TARGET_COL})  = {corr_pinpoint:.4f}")
print(f"corr(smoothed, {TARGET_COL})  = {corr_smoothed:.4f}")
print()

global_std = sample["pinpoint"].std()
local_spread = diff.abs()
print(f"global std of pinpoint predictions:        {global_std:.4f}")
print(f"mean |smoothed - pinpoint| (local spread):  {local_spread.mean():.4f}")
print(f"local spread as a fraction of global std:   {local_spread.mean() / global_std:.1%}")

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.hist(diff.clip(-0.05, 0.05), bins=100, color="#1baf7a", edgecolor="none")
ax.set_xlabel("smoothed - pinpoint (location_run_value units)")
ax.set_ylabel("pitch count")
ax.set_title("Per-pitch shift from target-averaging around the pitch's own location", loc="left", fontsize=10.5)
fig.tight_layout()
plt.show()

## Q2: aggregate. Does it change the season-level leaderboard?

Rebuild the (pitcher, pitch_type, season) aggregate and its 100+ calibration
under both versions, exactly as `location.py`'s own
`_ratio_calibration`/`_to_100_scale` do, and compare with Spearman rank
correlation. Same diagnostic `stuff.py`'s dev_log used to decide the
arsenal-contrast feature wasn't worth shipping (rho 0.998 there).

In [ ]:
def aggregate_and_calibrate(df, value_col):
    agg = (
        df.groupby([PITCHER_COL, PITCH_TYPE_COL, SEASON_COL], observed=True)
        .agg(mean_value=(value_col, "mean"), n_pitches=(value_col, "count"))
        .reset_index()
    )
    reliable = agg[agg["n_pitches"] >= stuff_mod.MIN_PITCHES_FOR_SCORE].copy()
    calibration = location_mod._ratio_calibration(reliable, [PITCH_TYPE_COL, SEASON_COL], "mean_value")
    agg = agg.merge(calibration, on=[PITCH_TYPE_COL, SEASON_COL], how="left")
    agg["plus"] = location_mod._to_100_scale(agg["mean_value"], agg)
    agg["reliable"] = agg["n_pitches"] >= stuff_mod.MIN_PITCHES_FOR_SCORE
    return agg


agg_pinpoint = aggregate_and_calibrate(sample, "pinpoint")
agg_smoothed = aggregate_and_calibrate(sample, "smoothed")

merged = agg_pinpoint.merge(
    agg_smoothed, on=[PITCHER_COL, PITCH_TYPE_COL, SEASON_COL], suffixes=("_pinpoint", "_smoothed")
)
reliable = merged[merged["reliable_pinpoint"] & merged["reliable_smoothed"]]
print(f"{len(reliable):,} reliable pitcher-pitch-type-seasons in this sample")

rho, _ = spearmanr(reliable["plus_pinpoint"], reliable["plus_smoothed"])
print(f"Spearman rank correlation (pinpoint vs. smoothed 100+ score): {rho:.4f}")

plus_diff = (reliable["plus_smoothed"] - reliable["plus_pinpoint"]).abs()
print(f"mean |plus_smoothed - plus_pinpoint|: {plus_diff.mean():.2f}")
print(f"pct moving by more than 2 points:     {(plus_diff > 2).mean():.1%}")
print(f"pct moving by more than 5 points:     {(plus_diff > 5).mean():.1%}")

rank_pinpoint = reliable["plus_pinpoint"].rank(ascending=False)
rank_smoothed = reliable["plus_smoothed"].rank(ascending=False)
top_15_pinpoint = set(rank_pinpoint[rank_pinpoint <= 15].index)
top_15_smoothed = set(rank_smoothed[rank_smoothed <= 15].index)
print(f"top-15 overlap: {len(top_15_pinpoint & top_15_smoothed)}/15")

## Synopsis

*(filled in after running the cells above)*